# Testing 2QAN on different coupling graphs and larger benchmark circuits

In this notebook, we benchmark the performance of py2qan across a range of circuit types (100 qubits each unless otherwise marked):

- Quantum Approximate Optimization Algorithm (QAOA)
- Quantum volume (QV) calculation
- Quantum Fourier transform (QFT)
- Square Heisenberg model Trotterized Hamiltonian simulation
- Quantum computational neural network (QCNN)
- PREPARE & SELECT on a 25 qubit GHZ state

### Circuit definitions
For the QAOA, QFT, Square Heisenberg, and QV circuits, we copy the corresponding OpenQASM 2 code from qiskit's benchpress library. For the QCNN and Prepare & Select circuits, we generate QASM 2 code using our own implementations in qiskit and cirq, decomposed into the restricted basis set:

In [1]:
folder = "./qasm_circuits/qasm2/"

qasm_files = [folder + file for file in [
    "benchpress/qaoa_barabasi_albert_N100_3reps_basis_rz_rx_ry_cx.qasm",
    "benchpress/qv_N100_12345_basis_rz_rx_ry_cx.qasm",
    "benchpress/qft_N100_basis_rz_rx_ry_cx.qasm",
    "benchpress/square_heisenberg_N100_basis_rz_rx_ry_cx.qasm",
    "ucc/prep_select_N25_ghz_basis_rz_rx_ry_h_cx.qasm",
    "ucc/qcnn_N100_7layers_basis_rz_rx_ry_h_cx.qasm"
    ]]

In [2]:
qasm_strings = []
for filename in qasm_files:
    with open(filename, "r") as file:
        qasm_strings.append(file.read())


We also generate different coupling graphs (in the form of lists of edges) to test circuit mapping and routing, specifically a tilted square lattice (sycamore) layout and a heavy-hex layout (as in IBM devices).

In [3]:
from cirq.devices import TiltedSquareLattice
from qiskit.transpiler import CouplingMap


def coords_to_labels(graph):
    """Convert lattice coordinates to node numbers (labels)."""
    labeled_edges = []
    labels = {str(node): n for n, node in enumerate(graph.nodes())}
    for edge in graph.edges:
        labeled_edges.append((labels[str(edge[0])], labels[str(edge[1])]))      
    return labeled_edges


def generate_tilted_square_coupling_list(width, height):
    """Returns coupling list for a tilted square lattice (sycamore layout)
    given the width and height of the device lattice."""
    tilted_square_layout = TiltedSquareLattice(width, height).graph
    return coords_to_labels(tilted_square_layout)


def generate_heavy_hex_coupling_list(distance):
    """Returns coupling list for a heavy-hex lattice given the distance of the
    device lattice."""
    return list(CouplingMap().from_heavy_hex(distance).get_edges())

coupling_lists = {"cl_sycamore": generate_tilted_square_coupling_list(15, 12), "cl_hhex": generate_heavy_hex_coupling_list(7)} 

In [4]:
import numpy as np
import time
# Import 2QAN compiler passes
from py2qan import BenchArch
from py2qan import HeuristicMapper
from py2qan import QuRouter
# Import qiskit 
import qiskit
from qiskit.transpiler import CouplingMap

Copied `qs_compiler` function from qiskit comparison notebook, removed the code comparing with qiskit for now.

In [5]:
def qs_compiler(qasm, coupling_map, layers=1, trials=1):
    qs_circ = None
    qs_swap = (0, 0) # the number of swaps in the format (#swaps,#swaps merged with circuit gate)
    qs_g2 = 0 # the number of two-qubit gates without decomposition
    # Perform qubit mapping, routing, and scheduling only, without gate decomposition
    for trial in range(trials):
        # Both QAP and Qiskit mappers output inital qubit maps randomly, 
        # one can run the mapper several times to achieve better compilation results
        # Initial qubit mapping 
        hmapper = HeuristicMapper(qasm, coupling_map=coupling_map)
        init_map = hmapper.run_qiskit(max_iterations=5)
        
        # init_map = {circuit qubit index:device qubit index}
        print('The initial qubit map is \n', init_map)
        start = time.time()
        # Routing and scheduling, takes init_map as input
        router = QuRouter(qasm, init_map=init_map, coupling_map=coupling_map)
        # For quantum simulation circuits, we assume each layer has the same time steps
        qs_circ, swaps = router.run(layers=layers, msmt='True')
        # qs_circ0 is the routed circuit without gate decomposition
        # swaps1 is a tuple=(#swaps,#swaps merged with circuit gate)
        end = time.time()
        print("Router run time: ", end - start)
        return qs_circ, swaps


Router seems to get stuck for these circuits- had to stop after ~5 min without any circuits routed.

In [6]:
for qs in qasm_strings:
    qs_compiler(qs, CouplingMap(coupling_lists["cl_sycamore"]))

The initial qubit map is 
 {0: 12, 1: 40, 2: 43, 3: 101, 4: 92, 5: 0, 6: 70, 7: 28, 8: 95, 9: 66, 10: 71, 11: 83, 12: 30, 13: 64, 14: 33, 15: 72, 16: 3, 17: 100, 18: 90, 19: 82, 20: 52, 21: 16, 22: 80, 23: 27, 24: 38, 25: 35, 26: 60, 27: 4, 28: 31, 29: 73, 30: 36, 31: 24, 32: 37, 33: 89, 34: 29, 35: 19, 36: 54, 37: 45, 38: 99, 39: 10, 40: 20, 41: 67, 42: 63, 43: 86, 44: 26, 45: 62, 46: 91, 47: 34, 48: 56, 49: 68, 50: 23, 51: 103, 52: 85, 53: 74, 54: 48, 55: 47, 56: 46, 57: 65, 58: 87, 59: 21, 60: 58, 61: 22, 62: 14, 63: 49, 64: 61, 65: 93, 66: 98, 67: 6, 68: 59, 69: 76, 70: 2, 71: 13, 72: 44, 73: 17, 74: 15, 75: 41, 76: 97, 77: 75, 78: 84, 79: 94, 80: 5, 81: 69, 82: 79, 83: 7, 84: 55, 85: 18, 86: 77, 87: 11, 88: 81, 89: 57, 90: 8, 91: 50, 92: 9, 93: 78, 94: 53, 95: 88, 96: 42, 97: 39, 98: 51, 99: 25, 100: 1, 101: 102, 102: 32, 103: 96}


KeyboardInterrupt: 

Router seems to get stuck for this coupling map- had to stop after ~5 min without the circuit being routed.

In [7]:
qs_compiler(qasm_strings[0], CouplingMap(coupling_lists["cl_hhex"]))

The initial qubit map is 
 {0: 42, 1: 43, 2: 44, 3: 45, 4: 46, 5: 47, 6: 48, 7: 70, 8: 72, 9: 109, 10: 110, 11: 111, 12: 112, 13: 113, 14: 114, 15: 85, 16: 79, 17: 11, 18: 3, 19: 83, 20: 8, 21: 52, 22: 12, 23: 84, 24: 76, 25: 51, 26: 4, 27: 36, 28: 82, 29: 62, 30: 9, 31: 98, 32: 7, 33: 50, 34: 100, 35: 59, 36: 78, 37: 38, 38: 75, 39: 99, 40: 30, 41: 107, 42: 53, 43: 18, 44: 37, 45: 31, 46: 77, 47: 97, 48: 0, 49: 65, 50: 63, 51: 81, 52: 108, 53: 54, 54: 1, 55: 13, 56: 27, 57: 25, 58: 2, 59: 90, 60: 94, 61: 28, 62: 66, 63: 29, 64: 101, 65: 80, 66: 49, 67: 69, 68: 17, 69: 71, 70: 5, 71: 74, 72: 21, 73: 34, 74: 105, 75: 33, 76: 95, 77: 14, 78: 103, 79: 26, 80: 56, 81: 10, 82: 32, 83: 57, 84: 55, 85: 40, 86: 73, 87: 19, 88: 89, 89: 96, 90: 60, 91: 106, 92: 64, 93: 20, 94: 6, 95: 102, 96: 41, 97: 24, 98: 68, 99: 35, 100: 15, 101: 86, 102: 16, 103: 87, 104: 88, 105: 91, 106: 22, 107: 92, 108: 23, 109: 93, 110: 104, 111: 39, 112: 58, 113: 61, 114: 67}


KeyboardInterrupt: 